# Session 1 · Adapting & Serving LLMs in Production
**Pharos-CY · Cyprus University of Technology**
**Instructors:** Yiannos Vyrides · Kyriakos-Evmenios N. Nomikos

Duration ≈ 2 hours · guided notebook. Every concept used below is explained in this notebook (and, with
interactive widgets, in the Session 1 lab pages) — no video or live narration is needed to follow it.

## What you will be able to do
1. Fine-tune an LLM with **QLoRA** without breaking it (SFT hyperparameters + failure modes).
2. Reason about **GRPO** and reward design — why we standardise per group, and how reward hacking happens.
3. Budget the **KV-cache** and choose a decoding strategy.
4. Understand efficient serving with **vLLM** (PagedAttention, LoRA hot-swap) and ground answers with a small **RAG** loop.

## Schedule
| Time | Part |
|------|------|
| 0:00 | Setup |
| 0:10 | A · Supervised fine-tuning + failure modes |
| 0:35 | QLoRA fine-tune (hands-on) |
| 1:00 | GRPO & reward design (concept + toy reward) |
| 1:20 | B · Decoding + KV-cache budgeting |
| 1:40 | vLLM serving (separate environment) + RAG (hands-on) |
| 1:55 | Wrap-up |

## Prerequisites
Python, PyTorch basics (tensors and a training loop), and transformer basics (tokens and attention).
No prior fine-tuning or LLM-serving experience is needed — the refresher below restates the four ideas everything else builds on.

## How to run
- **Colab:** `Runtime → Change runtime type → GPU`. A **T4** is enough — the notebook picks fp16 on a T4
  and bf16 on an Ampere-or-newer GPU automatically.
- **HPC / local:** one GPU, a fresh venv, then `pip install -r labs/requirements.txt` from the repository root.
- We use a small base model (`Qwen/Qwen2.5-0.5B-Instruct`) so every cell finishes in minutes. Where the
  course refers to a 13B, 4-bit base, that is called out inline as an *illustration*, not as this model.

> **One runtime, one job.** This notebook trains and generates with 🤗 Transformers on Colab's own
> PyTorch/CUDA stack. It deliberately does **not** install vLLM: vLLM ships its own torch/CUDA build and
> installing it here corrupts the runtime. The vLLM serving lab is a **separate environment** —
> see `labs/serve_vllm_rag.py` and the vLLM section near the end.


## Refresher · the four ideas everything below builds on

**1 · Tokens and next-token prediction.** Text is split into *tokens* (integer IDs). A causal language model reads the
tokens so far and outputs a *logit* for every entry of its vocabulary; a softmax turns the logits into a probability
distribution p_θ(x_t | x_<t) over the next token. Generation = pick a token, append it, repeat. θ denotes all the
model's trainable parameters.

**2 · One transformer layer.** The model stacks L identical layers. In each layer every token computes a Query, a Key
and a Value vector (Q = X·W_Q, K = X·W_K, V = X·W_V, with X ∈ ℝ^(n × d_model) for a sequence of n tokens) and attends
to the tokens before it:

  Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) · V

This runs in H heads in parallel (each of width d_h, so d_model = H · d_h) and is followed by a small MLP. The K and V
vectors of every past token, in every layer and head, are exactly what the **KV-cache** of Part B stores.

**3 · What training changes.** Training minimises a loss; for language models it is the cross-entropy
𝓛(θ) = − Σ_t log p_θ(x_t | x_<t). One *training step* = forward pass → loss → backward pass (the gradient ∂𝓛/∂θ) →
optimizer update θ ← θ − (learning rate) × gradient. A *batch* is the set of examples processed in one step; an *epoch*
is one pass over the whole dataset. Fine-tuning is this same loop, started from the pretrained θ and run on your data —
with LoRA, on a small set of extra parameters instead of θ itself.

**4 · Training vs inference.** During training the whole sequence is known and processed in one forward pass
(teacher forcing). During inference the model generates one token at a time and feeds its own output back — which is
why decoding strategy and the KV-cache only matter at serving time.


## Setup


In [ ]:
# ── Install (run this FIRST on a fresh GPU runtime) ───────────────────────────
# Two rules this cell follows, both learned from real Colab runs:
#
#  1. It never replaces Colab's preinstalled torch / torchvision / torchaudio / CUDA
#     wheels. Replacing them is what produced the "PyTorch compiled for CUDA 13.0 /
#     TorchAudio compiled for CUDA 12.8" breakage in an earlier version of this lab.
#  2. It does NOT install vLLM. vLLM pulls its own torch/CUDA build and would corrupt
#     this runtime. The vLLM lab lives in a separate environment (see the end of this
#     notebook and serve_vllm_rag.py).
#
# Versions are pinned because this course has been bitten twice by API drift:
# TRL renamed SFTConfig.max_seq_length -> max_length, and Transformers 5 removed
# TrainingArguments.warmup_ratio in favour of warmup_steps.
%pip install -q \
    "transformers==5.16.1" \
    "trl==1.12.0" \
    "peft==0.20.0" \
    "datasets==5.0.1" \
    "accelerate==1.14.0" \
    "sentence-transformers==6.0.1" \
    bitsandbytes
# bitsandbytes is intentionally left unpinned: it is a CUDA binary wheel and must stay
# free to match whichever CUDA build Colab's preinstalled torch was compiled against.

# ── One-time restart guard (this is the pyarrow fix) ──────────────────────────
# `datasets` requires pyarrow >= 21. If the Colab image ships an older pyarrow, pip
# upgrades it on disk while the OLD pyarrow is already loaded in this kernel (Colab
# imports pandas at startup, and importing pandas imports pyarrow). Mixing the two is
# what raises:
#     ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility
# A C-extension that was replaced under a running process can only be fixed by a fresh
# kernel — so we detect that exact situation and restart once, automatically.
# You never have to uninstall or reinstall anything by hand.
import importlib, importlib.metadata as _md, sys
importlib.invalidate_caches()

_stale = []
for _name in ("pyarrow", "numpy"):
    _loaded = sys.modules.get(_name)
    if _loaded is None:
        continue                      # not imported yet -> the new version will load
    try:
        if _md.version(_name) != getattr(_loaded, "__version__", None):
            _stale.append(_name)
    except _md.PackageNotFoundError:
        pass

if _stale:
    print("Upgraded under a running kernel:", ", ".join(_stale))
    print("Restarting the runtime once so the new binaries load cleanly.")
    print("→ When it comes back, run this cell again and continue straight down the notebook.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(True)
else:
    print("Environment ready — no restart needed.")

In [ ]:
# ── Imports, seeds, GPU detection ────────────────────────────────────────────
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── THE single source of truth for every dtype decision in this notebook ─────
# bf16 needs an Ampere-or-newer card (compute capability >= 8.0). A Colab T4 is
# sm_75, so it must train in fp16. Everything downstream — the 4-bit compute
# dtype, the weights we load, the Trainer's mixed precision, and the adapter
# dtype — is derived from `use_bf16` and nothing else.
if device == "cuda":
    major, minor = torch.cuda.get_device_capability()
    use_bf16 = major >= 8
    gpu_name = torch.cuda.get_device_name(0)
else:
    major, minor, use_bf16, gpu_name = 0, 0, False, None

compute_dtype = torch.bfloat16 if use_bf16 else torch.float16   # 4-bit compute + autocast
# CPU has no autocast here and no fp16 matmul kernels, so the CPU fallback loads fp32.
model_dtype = compute_dtype if device == "cuda" else torch.float32

import pyarrow, transformers, trl, peft, datasets, accelerate
print("torch       ", torch.__version__, "| CUDA:", torch.version.cuda, "| device:", device)
if device == "cuda":
    print("gpu         ", gpu_name, f"(sm_{major}{minor})")
    print("dtype plan  ", f"use_bf16={use_bf16} | compute_dtype={compute_dtype} | "
                          f"model_dtype={model_dtype} | bf16={use_bf16} fp16={not use_bf16}")
print("transformers", transformers.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
print("datasets    ", datasets.__version__)
print("accelerate  ", accelerate.__version__)
print("pyarrow     ", pyarrow.__version__)

if device == "cpu":
    print("\nNo GPU detected. Use Runtime -> Change runtime type -> GPU (a T4 is enough).")
    print("Every cell still runs on CPU, but the 4-bit path is skipped and training is slow.")

## Part A · Supervised fine-tuning (SFT)

**What SFT is.** Train a pretrained LLM on (prompt, response) pairs so it follows instructions and answers in the
right format. It is *imitation*: the same next-token cross-entropy as pretraining, but on a curated conditional
distribution. For a prompt x and a response y = (y₁, …, y_T):

  𝓛_SFT(θ) = − Σ_{t=1}^{T} log p_θ( y_t | x, y_<t )

- x = the serialised prompt (system + user turns, laid out by the model's **chat template**) · y_t = the t-th response
  token · θ = the trainable parameters (with LoRA: the adapter only).
- **Loss masking.** The sum runs over *response* tokens only: prompt tokens get the label −100 and contribute nothing.
  Below, TRL does this for us because we hand it `prompt` / `completion` fields (`completion_only_loss=True`).
- **Same template at training and inference.** TRL serialises every example with the tokenizer's own chat template,
  and the RAG cells later prompt the model through the same template (`apply_chat_template`). Training on one format
  and prompting with another looks like "the model got worse" even when the weights are fine.

SFT is behaviour shaping, not knowledge injection — facts come from retrieval (Part B).

**The hyperparameters that decide success**

| Knob | Sensible range | In this notebook |
|------|----------------|------------------|
| Learning rate | 1–2 × 10⁻⁵ | `learning_rate=2e-5` |
| Epochs | 1–3 (rarely > 5) | `num_train_epochs=1` |
| Warm-up | a few % of the steps | `warmup_steps=10` |
| Data diversity | many formats · styles · tasks | 800 mixed Dolly instructions |

> **Rule of thumb:** more epochs is *not* more learning — it is usually more **forgetting**.


### Two ways SFT quietly goes wrong

**1 · Catastrophic forgetting** — many epochs on a small or narrow dataset and the model overwrites broad skills with specialised ones.
*Fix:* 1–2 epochs · lower LR · mix 10–20% general data back into the set.

**2 · Style overfitting** — the model memorises the *form* of answers (always bullet points, a fixed length) without generalising the underlying skill.
*Fix:* fewer epochs · diverse formatting · augment the responses.

Both symptoms share one root cause: **too little variety, seen too many times.**


In [ ]:
from datasets import load_dataset

# A small, diverse instruction set keeps the demo fast and avoids style overfitting.
# Swap in your own (prompt, response) data for the real course.
N_TRAIN = 800
ds = load_dataset("databricks/databricks-dolly-15k", split="train").select(range(N_TRAIN))

def to_prompt_completion(ex):
    """One Dolly row -> TRL's conversational prompt/completion format.

    * prompt     = the user turn (the instruction, plus the row's context if it has one)
    * completion = the assistant turn (the reference response)

    Because both fields are lists of chat messages, SFTTrainer serialises the example with the
    tokenizer's own chat template (<|im_start|>user ... <|im_start|>assistant ...) -- exactly the
    format we prompt the model with later -- and computes the loss on the completion tokens
    only (loss masking: prompt tokens get the label -100).
    """
    instr = ex["instruction"]
    ctx = ex.get("context", "")
    prompt = instr if not ctx else f"{instr}\n\nContext:\n{ctx}"
    return {
        "prompt":     [{"role": "user",      "content": prompt}],
        "completion": [{"role": "assistant", "content": ex["response"]}],
    }

ds = ds.map(to_prompt_completion, remove_columns=ds.column_names)
print(ds)
print("-" * 60)
print("PROMPT     :", ds[0]["prompt"][0]["content"][:300])
print("COMPLETION :", ds[0]["completion"][0]["content"][:300])

## Low-Rank Adaptation (LoRA)

**What it is.** A linear layer computes h = W·x with a frozen pretrained weight W ∈ ℝ^(d_out × d_in). Instead of
updating W, LoRA adds a low-rank correction next to it:

  h = W·x + ΔW·x ,    ΔW = (α / r) · B·A ,    B ∈ ℝ^(d_out × r),  A ∈ ℝ^(r × d_in),  r ≪ min(d_in, d_out)

- **W is frozen** — all pretrained knowledge stays intact; only A and B receive gradients.
- **r** is the rank (`r=16` below); **α** is `lora_alpha` (32 below), so the update is scaled by α / r = 2.
  For square layers, write the update as h = Wx + BAx with B ∈ ℝ^(d×r), A ∈ ℝ^(r×d) — the same thing with
  d_in = d_out = d and the scale folded into B.
- **Init trick:** A random, B = 0 ⇒ ΔW = 0 at step 0 → training starts *exactly* at the base model.
- **Parameter count:** r·(d_in + d_out) instead of d_in·d_out per adapted matrix — for a square d×d matrix that is
  2·d·r instead of d². Tiny example: d_in = d_out = 8, r = 2 → 32 trainable numbers instead of 64. Real example: Qwen's
  `q_proj` is 896×896; with r = 16 that is 28,672 instead of 802,816 (3.6 %).
- **Where it goes:** `target_modules` below = the four attention projections (q, k, v, o) and the three MLP projections
  (gate, up, down) of every layer — 7 adapted matrices × 24 layers.

**Why LoRA wins twice**
1. *Regulariser* — very few free parameters: a structural bottleneck that resists overfitting on a small dataset.
2. *Serving superpower* — one base copy on the GPU + tiny hot-swappable adapters (≈ 0.1 GB each for a 13B base at
   r = 16) instead of a full model per behaviour (vLLM `enable_lora`, Part B).

**QLoRA** = the same adapters on top of a base whose frozen weights are stored in **4-bit NF4** and dequantised on the
fly for each matmul: h = dequant(Ŵ)·x + ΔW·x. The base shrinks ≈ 4× vs FP16, the adapters stay in higher precision and
the optimizer only tracks the adapters — that is what fits a 13B model on one 16 GB card.


In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE = "Qwen/Qwen2.5-0.5B-Instruct"   # teaching model. 13B numbers quoted later are illustrations.

# 4-bit NF4 quantisation -- this is what lets a much larger base fit on one 16 GB card.
# compute_dtype follows the GPU: bfloat16 on Ampere+, float16 on a T4.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb if device == "cuda" else None,   # 4-bit needs CUDA
    dtype=model_dtype,          # explicit! Transformers 5 defaults to dtype="auto", which
                                # reads Qwen's config (torch_dtype: bfloat16) and would load
                                # the whole model in bf16 even on a T4 that cannot train it.
    device_map="auto" if device == "cuda" else None,   # "auto" is for placing/sharding on
                                # GPUs; on CPU it can decide to disk-offload layers to the
                                # meta device, which then breaks the Trainer.
)
model = prepare_model_for_kbit_training(model)

# LoRA on the attention + MLP projections of every layer, rank r << d.
# lora_alpha / r = 32 / 16 = 2 is the scale applied to the low-rank update B @ A.
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# 1 epoch, LR 2e-5, cosine scheduler + 10 warmup steps.
# API note: this is TRL's current surface -- max_length (not max_seq_length) and
# warmup_steps (Transformers 5 removed warmup_ratio).
# completion_only_loss=True is the loss masking from Part A: TRL builds a mask from the
# prompt/completion fields, so only the response tokens are trained on.
trainer = SFTTrainer(
    model=model,
    peft_config=lora,
    train_dataset=ds,
    args=SFTConfig(
        output_dir="./adapter",
        num_train_epochs=1,
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        bf16=use_bf16,
        fp16=(device == "cuda" and not use_bf16),
        logging_steps=10,
        completion_only_loss=True,
        max_length=1024,
        report_to="none",
    ),
)

# -- Reconcile TRL's own dtype override (this is what breaks QLoRA on a T4) ---
# SFTTrainer.__init__ ends with, unconditionally, for any 4-bit/8-bit model:
#
#     if _is_quantized_model:
#         for param in model.parameters():
#             if param.requires_grad:
#                 param.data = param.data.to(torch.bfloat16)
#
# It follows the QLoRA paper's bf16 recommendation without asking whether the GPU
# can do bf16. On a T4 we train with fp16 + GradScaler, and the unscale kernel has
# no bfloat16 implementation, so the very first optimizer step dies with:
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not
#                        implemented for 'BFloat16'
# fp32 adapter weights are the normal master-weight dtype for fp16 mixed precision,
# so put them back. On an Ampere+ GPU use_bf16 is True and TRL's cast is left alone.
if not use_bf16:
    recast = sum(1 for p in trainer.model.parameters()
                 if p.requires_grad and p.dtype == torch.bfloat16)
    for p in trainer.model.parameters():
        if p.requires_grad and p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32)
    print(f"fp16 path: recast {recast} trainable tensors bfloat16 -> float32")

trainable_dtypes = sorted({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})
print("trainable parameter dtypes:", trainable_dtypes)
assert use_bf16 or "torch.bfloat16" not in trainable_dtypes, (
    "bfloat16 trainable parameters on a GPU whose GradScaler cannot unscale them"
)

In [ ]:
# What one training example looks like AFTER the trainer has prepared it: the chat template
# has been applied, and `labels` carries -100 on every token the loss must ignore (loss masking).
ex = trainer.train_dataset[0]
ids, labels = ex["input_ids"], ex["labels"]
n_masked = sum(1 for l in labels if l == -100)
print(tok.decode(ids))
print("-" * 60)
print(f"{len(ids)} tokens in the sequence; {n_masked} prompt/template tokens are masked (label -100),")
print(f"the loss is computed on the remaining {len(ids) - n_masked} -- the assistant turn:")
print(repr(tok.decode([t for t, l in zip(ids, labels) if l != -100])))

In [ ]:
trainer.train()

trainer.model.save_pretrained("./adapter")   # a few MB of adapter, not the full base
import os
print("Saved LoRA adapter to ./adapter:", sorted(os.listdir("./adapter")))

In [ ]:
# Sanity check: the exact API surface this notebook depends on.
# If a future TRL/Transformers release renames one of these, this cell says so
# immediately instead of failing three cells later.
import inspect, os
import trl
from trl import SFTConfig

params = inspect.signature(SFTConfig).parameters
print("TRL version:                     ", trl.__version__)
print("trainer.model type:              ", type(trainer.model).__name__)
print("SFTConfig has max_length:        ", "max_length" in params, " (old name: max_seq_length)")
print("SFTConfig has warmup_steps:      ", "warmup_steps" in params, " (removed: warmup_ratio)")
print("SFTConfig has completion_only_loss:", "completion_only_loss" in params)
print("adapter written:                 ", os.path.exists("./adapter/adapter_model.safetensors"))

### ✎ Exercise — how big is the adapter?
Each adapted matrix W ∈ ℝ^(d_out × d_in) gets r·(d_in + d_out) trainable parameters (2·d·r when it is square).
Using the model's config — hidden size 896; 14 query heads and 2 key/value heads of width 64; MLP width 4864;
24 layers — add up the seven target modules of one layer, multiply by the number of layers, and compare with
`print_trainable_parameters()` in the cell below. Then do the same for a 13B model (d = 5120, 40 heads of width 128
with no GQA, MLP width 13824, 40 layers).


In [ ]:
# ✓ Solution
# print_trainable_parameters() lives on the PEFT model. SFTTrainer wraps the model we
# passed in, so call it through the trainer -- plain `model.print_trainable_parameters()`
# raises AttributeError, because `model` here is still the un-wrapped Transformers model.
trainer.model.print_trainable_parameters()

def lora_params(d_model, n_heads, n_kv_heads, head_dim, d_mlp, n_layers, r):
    """r*(d_in + d_out) summed over the 7 target modules of one layer, times n_layers."""
    d_q, d_kv = n_heads * head_dim, n_kv_heads * head_dim
    per_layer = r * ((d_model + d_q)          # q_proj      : d_model -> d_q
                     + 2 * (d_model + d_kv)   # k_proj, v_proj : d_model -> d_kv (GQA makes these small)
                     + (d_q + d_model)        # o_proj      : d_q -> d_model
                     + 2 * (d_model + d_mlp)  # gate_proj, up_proj : d_model -> d_mlp
                     + (d_mlp + d_model))     # down_proj   : d_mlp -> d_model
    return per_layer * n_layers

cfg = trainer.model.config
r = lora.r
n_formula = lora_params(cfg.hidden_size, cfg.num_attention_heads, cfg.num_key_value_heads,
                        cfg.hidden_size // cfg.num_attention_heads, cfg.intermediate_size,
                        cfg.num_hidden_layers, r)
n_trainable, n_all = trainer.model.get_nb_trainable_parameters()   # PEFT's own count (4-bit aware)
print(f"{BASE}: r={r}  formula={n_formula:,}  trainable={n_trainable:,}  "
      f"-> {100*n_trainable/n_all:.2f}% of all parameters")

# --- Illustration: a 13B-class model, NOT the 0.5B model loaded above ------------
# d = 5120 is the hidden size of a 13B model (the production base referenced by the
# course): 40 heads of width 128, no GQA, MLP width 13824, 40 layers.
n_13b = lora_params(5120, 40, 40, 128, 13824, 40, r)
print(f"13B (illustration): LoRA params={n_13b:,}  ~{n_13b*2/1e9:.2f} GB in FP16  "
      f"-> {100*n_13b/13e9:.2f}% of 13e9")

# For one square d x d matrix the formula collapses to 2*d*r (the form used in the course):
for label, d in [("13B q_proj (illustration)", 5120), ("Qwen2.5-0.5B q_proj (loaded)", 896)]:
    print(f"{label}: dense d*d = {d*d:>10,}   LoRA 2*d*r = {2*d*r:>8,}"
          f"  ->  {100*2*d*r/(d*d):.2f}% of the dense update")

# --- Illustration: why hot-swappable adapters win at serving time ---------------
# A 13B model in FP16/BF16 is ~13e9 * 2 bytes ~= 26 GB of weights; the adapter above is ~0.1 GB.
base_gb, adapter_gb = 26, 0.1
print(f"\n3 full 13B copies: {3*base_gb} GB   vs   1 base + 3 adapters: {base_gb + 3*adapter_gb:.1f} GB")

## From imitation to optimisation — GRPO

**Why.** SFT can only *imitate* data; it cannot directly optimise **correctness** or **safety**, which do not reduce to
clean (input, output) pairs. So: light SFT (1 epoch) for a sensible starting policy, then **GRPO** (RL) to sharpen it.

**What.** For each prompt q_i, sample a *group* of G answers y_{i1}, …, y_{iG} from the current policy π_θ and score
each one with a reward r_{ij} = R(q_i, y_{ij}). Standardise the rewards **within the group**:

  μ_i = (1/G) Σ_j r_{ij} ,    σ_i = √( (1/G) Σ_j (r_{ij} − μ_i)² ) ,    A_{ij} = ( r_{ij} − μ_i ) / ( σ_i + ε )

A_{ij} is the **advantage** of answer j: how much better it is than its siblings. The group itself provides the
baseline, so no separate value network (critic) is needed; ε is a small constant that avoids dividing by zero.

**The update**. The policy is moved to make high-advantage answers more likely, through a PPO-style
clipped probability ratio ρ_{ij} = π_θ(y_{ij} | q_i) / π_θold(y_{ij} | q_i), while a KL penalty keeps it near the SFT
reference π_ref:

  𝓛(θ) = E[ min( ρ_{ij} A_{ij} ,  clip(ρ_{ij}, 1 − ε_clip, 1 + ε_clip) A_{ij} ) ] − β · KL( π_θ ‖ π_ref )

You will not run this loop today (it needs `trl`'s `GRPOTrainer` and a GPU budget); the cells below isolate the two
ideas that decide whether it works.

**Why divide by σ_i?** It gives every group's advantages a comparable magnitude regardless of that group's reward
scale. Tiny example — two prompts with six sampled answers each:
- a safety-critical question, rewards 0.5, 5.0, 1.0, 4.5, 2.0, 5.0 → μ = 3.0, σ ≈ 1.9
- a simple definition, rewards 4.0, 4.2, 4.5, 4.1, 4.3, 4.0 → μ ≈ 4.18, σ ≈ 0.18

Subtract the mean only and the best definition answer gets an advantage of +0.32 while the best safety answer gets
+2.0 — the optimiser chases the safety group and the definition prompt teaches almost nothing. Divide by σ and the
best answers land at +1.8 and +1.1: every prompt type gets an equal vote in the update. Run the cell to see both rows.


In [ ]:
# Group-relative advantage: standardise rewards WITHIN each prompt's group.
import numpy as np

def group_advantage(rewards, eps=1e-6, divide_by_sigma=True):
    """A_ij = (r_ij - mu_i) / (sigma_i + eps) for one prompt's group of rewards."""
    rewards = np.asarray(rewards, float)
    centred = rewards - rewards.mean()
    return centred / (rewards.std() + eps) if divide_by_sigma else centred

groups = {
    "safety-critical prompt": [0.5, 5.0, 1.0, 4.5, 2.0, 5.0],   # answers range excellent -> dangerous
    "simple definition     ": [4.0, 4.2, 4.5, 4.1, 4.3, 4.0],   # every answer is roughly a 4 or 5
}
for name, r in groups.items():
    r = np.asarray(r, float)
    print(f"{name}: mean={r.mean():.2f}  std={r.std():.2f}")
    print("   mean-subtracted only:", np.round(group_advantage(r, divide_by_sigma=False), 2))
    print("   standardised (GRPO) :", np.round(group_advantage(r), 2))

safety, define = groups.values()
raw = [np.abs(group_advantage(g, divide_by_sigma=False)).max() for g in (safety, define)]
std = [np.abs(group_advantage(g)).max() for g in (safety, define)]
print(f"\nlargest |advantage|, safety vs definition -- without /sigma: {raw[0]:.2f} vs {raw[1]:.2f}"
      f" ({raw[0]/raw[1]:.0f}x)  |  with /sigma: {std[0]:.2f} vs {std[1]:.2f}")
# Without the division the definition prompt's signal is ~8x smaller, so the optimiser ignores it.

### Reward design & reward hacking

**Reward hacking** = the policy optimises the metric, not the intent. Collapse correctness and safety into one number,
r = α·r_correct + (1 − α)·r_safety, and a perfectly correct but dangerous answer (correctness 5, safety 1) scores 3.8
with α = 0.7 — it looks "fine" and the failure is invisible. Keep the signals **separate**, enforce a **floor** on the
non-negotiable one (e.g. r_safety ≥ 4, otherwise a hard penalty), and you can watch *which* dimension is being gamed.
The KL term keeps the policy near the SFT reference but on its own may not stop hacking — add per-signal limits beyond KL.


In [ ]:
# Illustrative reward (not a full GRPO loop): keep signals separate, enforce a safety floor,
# then standardise per group exactly as above.
def reward(correctness, safety, alpha=0.7, safety_floor=4.0):
    r = alpha*correctness + (1-alpha)*safety
    return np.where(safety < safety_floor, r - 5.0, r)     # hard penalty below the floor

drug   = reward(np.array([5,4,1,2]), np.array([5,2,5,1]))   # large spread; answer 2 is fairly correct but unsafe
define = reward(np.array([5,5,4,5]), np.array([5,5,5,5]))   # tiny spread

print("drug   rewards:", np.round(drug, 2), " spread:", round(float(drug.std()), 2))
print("define rewards:", np.round(define, 2), " spread:", round(float(define.std()), 2))
print()
print("drug   group advantages:", np.round(group_advantage(drug), 2))
print("define group advantages:", np.round(group_advantage(define), 2))
# Answer 2 of the drug prompt (correctness 4, safety 2) would score 3.4 in the blended reward;
# the floor pushes it to -1.6, so the policy learns "unsafe is unsafe", not "80% correct is fine".
# Both groups still contribute comparably because of the division by sigma.

## Part B · Serving at scale

### Decoding strategy — how the next token is picked
At every step the model gives logits ℓ over the vocabulary and p = softmax(ℓ). The *decoding rule* decides what to do
with p:

| Rule | What it does | Character | Where you meet it here |
|------|--------------|-----------|------------------------|
| **Greedy** | take the argmax of p at every step | deterministic, fast, can loop | `do_sample=False` in the RAG cells |
| **Beam search** | keep the k best partial sequences, maximise P(sequence) | deterministic, length-biased | translation, factual QA |
| **Temperature T** | p ∝ exp(ℓ / T): T < 1 sharpens, T > 1 flattens | a knob applied before sampling | `temperature=0.2` in the vLLM example |
| **Top-k** | sample from the k most probable tokens | stochastic | `top_k=…` |
| **Nucleus (top-p)** | sample from the smallest set whose cumulative probability ≥ p | stochastic, adapts to the shape of p | `top_p=0.9` in the vLLM example |

- Beam search suits targeted tasks (translation, factual QA); sampling suits open-ended text (chat, creative writing).
- **For code:** nucleus (p ≈ 0.95, T ≈ 0.8) + *best-of-N with test execution* beats beam — beam collapses to identical
  boilerplate, while one correct sampled candidate is enough.

### The memory that surprises everyone — the KV-cache
**Why it exists.** Attention needs the Key and Value vectors of every earlier token. Recomputing them for each new
token would make generation quadratic, so a server keeps them in memory: one K and one V vector per token, per layer,
per key/value head. That memory is the KV-cache, and it grows with the sequence:

  M_KV ≈ 2 · S · L · H_kv · d_h · b/8   bytes per sequence

S = sequence length (prompt + generated tokens) · L = layers · H_kv = **key/value** heads (equal to the H attention
heads unless the model uses GQA/MQA) · d_h = head dim · b = bits per value (16 for FP16). The factor 2 = one Key tensor
+ one Value tensor. For B concurrent sequences, multiply by B.

**Worked example** — a 13B model (L = 40, H = 40, d_h = 128), FP16, S = 4096:
2 · 4096 · 40 · 40 · 128 · 2 bytes = 3,355,443,200 bytes ≈ 3.36 GB — for a *single* sequence, on top of ≈ 26 GB of
weights. 64 concurrent users would need ≈ 215 GB of cache; that is why long-context serving is dominated by the cache,
not the weights.

Reductions: **INT8** KV (b = 8: ~50% off, small accuracy cost); **MQA / GQA** share K,V heads across query heads so
that H_kv ≪ H (a bigger win, needs an architecture change — Qwen2.5-0.5B has 14 query heads but only 2 KV heads).


In [ ]:
def kv_cache_gb(S, L, H_kv, d_h, bits):
    """KV-cache size for ONE sequence, in GB. H_kv = number of key/value heads."""
    return (2 * S * L * H_kv * d_h * (bits/8)) / 1e9

# Worked example from the course: a 13B model (40 layers, 40 heads, head_dim 128,
# no GQA -> 40 KV heads), FP16.
print("13B FP16, S=4096:", round(kv_cache_gb(4096, 40, 40, 128, 16), 2), "GB / sequence")

# Exercise: what does INT8 do to that number, and what about 64 concurrent sequences?
print("13B INT8, S=4096:", round(kv_cache_gb(4096, 40, 40, 128, 8), 2), "GB / sequence")
print("FP16 x 64 seqs  :", round(64*kv_cache_gb(4096, 40, 40, 128, 16), 1), "GB")
print()

# The model you just fine-tuned, read straight off its config — GQA in action.
cfg = trainer.model.config
L      = cfg.num_hidden_layers
H_q    = cfg.num_attention_heads
H_kv   = getattr(cfg, "num_key_value_heads", H_q)      # GQA: fewer KV heads than query heads
d_h    = getattr(cfg, "head_dim", None) or cfg.hidden_size // H_q
print(f"{BASE}: layers={L}, query heads={H_q}, KV heads={H_kv}, head_dim={d_h}")
print("  with GQA        :", round(kv_cache_gb(4096, L, H_kv, d_h, 16) * 1e3, 1), "MB / sequence")
print("  if it had no GQA:", round(kv_cache_gb(4096, L, H_q,  d_h, 16) * 1e3, 1), "MB / sequence")
print(f"  -> GQA saves {H_q // H_kv}x on the cache, for free.")

### High-throughput serving — vLLM & PagedAttention

Naive serving reserves *contiguous* KV memory for the maximum sequence length, per request — most of it sits unused,
memory fragments, GPUs idle. **PagedAttention** pages the KV-cache like OS virtual memory (small non-contiguous blocks,
allocated on demand, near-zero waste); **continuous batching** lets new requests join a running batch as others finish
instead of waiting for the whole batch to end. The vLLM paper (Kwon et al., 2023) reports ≈ 24× the throughput of a
naive 🤗 `generate` loop and < 4% KV waste, and `enable_lora` hot-swaps adapters per request — the LoRA serving win.

**Scaling across GPUs:** Data-parallel (model fits — split the batch, FSDP) → Tensor-parallel (a single layer is too big) → Pipeline-parallel (the model is too deep). Start with FSDP; add tensor/pipeline only when a layer or depth will not fit.

> ⚠️ **vLLM runs in a separate environment — not in this runtime.**
> vLLM installs its own torch/CUDA build. Running `pip install vllm` in this notebook replaces
> Colab's preinstalled PyTorch/CUDA wheels and breaks the training stack you just used
> (the symptom is a "PyTorch compiled for CUDA X / TorchAudio compiled for CUDA Y" import error).
> The next cell therefore **prints** the serving code rather than running it. To actually run it,
> use `labs/serve_vllm_rag.py` on a dedicated vLLM machine/container — the adapter you saved to
> `./adapter` is all you need to carry across.


In [ ]:
# Production-serving reference (vLLM) — SEPARATE ENVIRONMENT, not executed here.
#
# Run this on a dedicated vLLM box/container with a matched CUDA / PyTorch / vLLM
# install, exactly as it appears in serve_vllm_rag.py. Do NOT `pip install vllm`
# into this Colab training runtime.
#
# dtype="float16" and gpu_memory_utilization=0.80 keep it inside a T4-class card;
# on an Ampere+ GPU you can use dtype="bfloat16" and push utilisation higher.
vllm_example = r"""
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",                 # "bfloat16" on Ampere or newer
    gpu_memory_utilization=0.80,
    enable_prefix_caching=True,      # reuse the KV-cache across shared prefixes
    enable_lora=True,                # hot-swap adapters per request
    max_model_len=2048,
    # tensor_parallel_size=2,        # uncomment for 2-GPU tensor parallelism
)

sp = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=128)
out = llm.generate(
    ["Explain LoRA in two sentences."],
    sp,
    lora_request=LoRARequest("session1-adapter", 1, "./adapter"),
)
print(out[0].outputs[0].text)
"""

print(vllm_example)
print("Reference only — see serve_vllm_rag.py. This runtime keeps Colab's training stack intact.")

### Grounding answers with RAG

Fine-tuning teaches a *skill*; it does not install *facts*. For grounded, citable answers you retrieve relevant
context at query time and instruct the model to use **only** that context — or admit it does not know.

**How retrieval works here.** An *embedding model* (`all-MiniLM-L6-v2`) maps any text to a 384-dimensional vector
such that similar meanings give nearby vectors. The corpus is embedded once; at query time the question is embedded,
its **cosine similarity** to every corpus vector is computed, and the top-k passages are pasted into the prompt with
citation tags [0], [1], … A vector database (FAISS, Milvus) does the same thing at scale.

**Three guardrails — retrieval alone does not ground a model.**
1. **Retrieval gate.** Top-k always returns *something*, even for a question the corpus cannot answer: for
   "Who painted the Mona Lisa?" the best passage scores cosine ≈ 0.1, pure noise. Passages below `MIN_SIM` are dropped,
   so the prompt says "(no relevant passages found)" instead of presenting filler with citation tags. Measured on this
   corpus, passages that contain the answer score ≈ 0.45–0.85 and unrelated ones ≈ 0.0–0.2, so the gate sits at 0.35.
2. **Copy-friendly decoding.** Greedy decoding, and `repetition_penalty=1.0`: Qwen's shipped generation config applies a
   1.1 penalty to every token already present in the prompt — even with sampling switched off — which fights the
   verbatim copying a grounded answer needs.
3. **Citation check.** The displayed answer is the model's output only if it cites a passage that was actually in the
   context; a bare "unknown" or a citation of a passage that does not exist is displayed as "unknown". The raw output is
   printed too, so you can see when the model tried to bluff.

The cells below run the loop **on the model you just fine-tuned**: a question the corpus answers about the course
material, a question it answers from a single geography passage, and a question it cannot answer. No vLLM needed.


In [ ]:
# Minimal grounded QA (RAG): retrieve -> gate -> build a CITED context -> generate -> check the citation.
# Tiny in-memory store so it runs anywhere; swap for FAISS / Milvus at scale.
import re
from sentence_transformers import SentenceTransformer, util

corpus = [
    "LoRA freezes W and learns a low-rank BA update; only 2*d*r parameters train.",
    "PagedAttention stores the KV-cache in non-contiguous blocks to cut waste.",
    "GRPO standardises rewards within a group so different reward scales contribute comparably.",
    "Dividing by the group standard deviation puts every prompt group on the same scale, so a "
    "high-variance group cannot dominate the update and a low-variance group still teaches the model.",
    "Nucleus sampling draws from the smallest set of tokens whose probability mass exceeds p.",
    "Nicosia is the capital of Cyprus.",
]

embed = SentenceTransformer("all-MiniLM-L6-v2")
C = embed.encode(corpus, convert_to_tensor=True)

# Retrieval gate. On this corpus a passage that contains the answer scores cos_sim ~0.45-0.85 and an
# unrelated one ~0.0-0.2 (show() prints the scores). Below the gate a passage is not evidence and
# must not enter the prompt: top-k always returns *something*, and handing the model irrelevant
# text with a citation tag is an invitation to bluff "<something> [0]".
MIN_SIM = 0.35

def retrieve(query, k=2, min_sim=MIN_SIM):
    """Top-k passages by cosine similarity that pass the gate -> [(text, score)] in rank order."""
    q = embed.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q, C)[0]
    top = scores.topk(min(k, len(corpus)))
    return [(corpus[i], float(s)) for s, i in zip(top.values, top.indices) if float(s) >= min_sim]

def build_prompt(query, docs):
    """Context (or an explicit 'nothing found') first, rules AFTER it, then two worked examples.

    Prompt order matters for a 0.5B model: with the rules placed before the context it ignores
    the citation rule, and with rules-but-no-examples it answers "unknown" to everything. Rules
    last + two examples is what actually produced cited answers here -- test your own wording,
    do not assume it transfers.
    """
    context = "\n".join(f"[{i}] {d}" for i, (d, _) in enumerate(docs)) or "(no relevant passages found)"
    return (
        f"Context:\n{context}\n\nQuestion: {query}\n\n"
        "Rules:\n"
        "1. Answer in one or two sentences using ONLY the context above.\n"
        "2. Finish with the citation of the item you used, e.g. [0].\n"
        "3. If the context does not contain the answer, reply with exactly: unknown\n\n"
        'Example of a grounded answer: "Beam search keeps k candidates. [1]"\n'
        'Example when the context does not help: "unknown"\n\nAnswer:'
    )

def answer(query, k=2, max_new_tokens=96):
    """retrieve -> gate -> prompt -> generate -> check the citation. Uses the adapter already in memory.

    Returns (prompt, displayed_answer, raw_model_output). The displayed answer is the raw output only
    when it cites a passage that was really in the context; anything else -- a bare "unknown", or a
    citation of a passage that does not exist -- is displayed as "unknown".
    """
    docs = retrieve(query, k)
    prompt = build_prompt(query, docs)
    chat = [{"role": "user", "content": prompt}]
    # Same chat template the trainer applied to the training data (no train/inference mismatch).
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(trainer.model.device)

    trainer.model.eval()
    trainer.model.config.use_cache = True          # the Trainer turns this off for training
    with torch.no_grad():
        generated = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                       # greedy: deterministic for a teaching demo
            repetition_penalty=1.0,                # Qwen ships 1.1, which penalises copying prompt tokens
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    raw = tok.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    cited = [int(i) for i in re.findall(r"\[(\d+)\]", raw)]
    grounded = bool(cited) and all(i < len(docs) for i in cited) and not raw.lower().startswith("unknown")
    return prompt, (raw if grounded else "unknown"), raw

def show(query, k=2):
    """Trace one question: what was retrieved (and whether it passed the gate), the raw output, the displayed answer."""
    prompt, final, raw = answer(query, k)
    print("Q:", query)
    for i, (d, s) in enumerate(retrieve(query, k, min_sim=-1.0)):     # unfiltered top-k, for the trace
        print(f"   top-{i} cos_sim={s:.2f} {'-> context [' + str(i) + ']' if s >= MIN_SIM else '-> below gate, dropped'}: {d}")
    print("   raw model output:", repr(raw))
    print("   displayed answer:", final)

# Case B -- a question the course-material passages answer
query = "Why does GRPO divide by the group standard deviation?"
prompt, grounded, raw = answer(query)
print("=== Prompt sent to the model ===")
print(prompt)
print("\n=== Displayed answer ===")
print(grounded)

In [ ]:
# Case A -- a question answered by a single passage: the fact must come from the context, with its citation.
show("What is the capital of Cyprus?")

In [ ]:
# Case C -- the other half of grounding: a question the corpus cannot answer must end in "unknown".
show("Who painted the Mona Lisa?")
# Nothing scores above the gate, so the prompt says "(no relevant passages found)". A small model may still
# bluff -- e.g. "... Leonardo da Vinci. [0]" -- but a citation of a passage that does not exist fails the
# citation check, so the displayed answer is "unknown". Retrieval alone does not ground a model; the gate,
# the prompt and the citation check all have to.

### ✎ Exercise
1. Call `show(query, k=3)` and see whether the extra retrieved item helps or dilutes the answer.
2. Delete the two `Example ...` lines from `build_prompt` and re-run the three questions. A 0.5B model
   stops citing without them — grounding is as much prompt engineering as retrieval.
3. Move the `Rules:` block *above* the context and re-run. Watch the behaviour change again.
4. Set `MIN_SIM = 0` and re-run the Mona Lisa question: the two nearest (irrelevant) passages now enter the
   prompt with citation tags. Does the model still say unknown, or does it cite them? That is the failure the gate
   prevents.
5. Ask `show("What is the capital of Greece?")`. The Cyprus passage passes the gate (cos ≈ 0.5) but does not answer
   the question — the hard case, where only the model's discipline protects you. Small models often reply with the
   Cyprus sentence; a stronger model, a higher gate or a reranker are the remedies.
6. Add one of your own facts to `corpus`, then ask a question only that fact can answer.

## Wrap-up
- Fewer epochs + diverse data beat brute-force SFT; **LoRA is a regulariser *and* a serving superpower.**
- Know your **KV-cache** budget and page it (vLLM); pick decoding by task; scale with FSDP first.
- Ground answers with retrieval **plus** a gate, copy-friendly decoding and a citation check — and let the model say unknown.
- Keep the **training runtime** and the **vLLM serving runtime** separate — one installs over the other.


